**cantods_diagnostics**

This notebook is used to plot CanTODS diagnostics that have been processed using cantods_rtd, either at run time, or a posteriori.
Whereas the netCDF files are saved at the default temporal resolution (5 days), different averaging is able to be performed here.

Requires python3; tested with the py3_analysis_v2 environment and joi000's personal environment.

Written by Jonathan Izett (October 2024)

In [ ]:
# User options

# list of runids to look at
# e.g., ['jgi-ctds-multi17']
runids=['RUNID']

# list of paths where runids are found
# e.g., ['/home/joi000/site5/analysis/CanTODS/']
paths=['RTDPATH']

# initial year
year0=[YEAR0]

# variable(s) to plot
# either a list of variable names, or 'all' (and list determined from first runid)
# variable names include depth and region, e.g., siarea, siarea_labsea, T_4000m_pac, etc.
variables=['all']

# temporal frequency for plots (resamples dataset)
# e.g., freqs2plot=['default','1M','Q-MAR','1Y']
#       plots file default, 1-month avg, seasonal average, and 1Y average
freqs2plot=['default','1M','Q-MAR','1Y']

# resample & plot max, min, or mean
resampleType='mean'

# plot the mean annual cycle over all years
plotAnnual=True

In [ ]:
# Import relevant modules
import datetime as dt
import glob
import matplotlib.pyplot as plt
import numpy as np
import os
import xarray as xr

#############
# FUNCTIONS #
#############

def varList(varName,path0,run0):
    # search for all variables
    if varName=='all':
        flist=glob.glob(os.path.join(path0,f'{run0}_*_timeseries.nc'))
    else:
        flist=glob.glob(os.path.join(path0,f'{run0}_{varName}_timeseries.nc'))
    vars2append=[]
    if len(flist) > 0:
        for fF in flist:
            vName=''
            vSplit=os.path.basename(fF).split('_')
            for iV,vV in enumerate(vSplit):
                if iV > 0 and vV != 'timeseries.nc':
                    vName+=f'{vV}_'
            vars2append.append(vName[0:-1])
    return vars2append

def annualCycle(dset,var,lcol,rName,axs):
    """
    Plot the mean annual cycle.
    """
    # resample dataset to monthyl frequency
    if resampleType=='min':
        rds=ds.drop_duplicates(dim='time',keep='last').sortby('time').resample(time='1M',skipna=True).min()
    elif resampleType=='max':
        rds=ds.drop_duplicates(dim='time',keep='last').sortby('time').resample(time='1M',skipna=True).max()
    else:
        rds=ds.drop_duplicates(dim='time',keep='last').sortby('time').resample(time='1M',skipna=True).mean()

    # get mean of each month
    mnths=rds.time.dt.month
    aMean=np.full(12,np.nan)
    for mM in range(1,13):
        aMean[mM-1]=np.nanmean(rds[var].values[mnths==mM])
    
    # plot
    axs.plot(range(1,13),aMean,color=lcol,linewidth=1,label=rName)

#######
# RUN #
#######

# close existing figures
plt.close('all')

# get list of variables to plot from first runid if not given
vars2plot=[]
for vV in variables:
    vars2plot.extend(varList(vV,paths[0],runids[0]))

if len(vars2plot) == 0:
    print('No variables to plot!')
else:
    # loop through each variable
    nF=0
    for iV,var in enumerate(vars2plot):
        print(var)
        if rsType is not None:
            resampleType=rsType
        elif ('T' in var) or ('S' in var):
            resampleType='mean'
        elif ('si' in var) or ('MLD' in var):
            resampleType='max'
        # create empty figures
        fig={}; ax={}
        for iF,freq in enumerate(freqs2plot):
            fig[f'{var}_{freq}']=plt.figure(nF+1); nF+=1
            ax[f'{var}_{freq}']=plt.gca()
            fig[f'{var}_{freq}'].set_figwidth(1.25*fig[f'{var}_{freq}'].get_figwidth())
            plt.ylabel(var)
            if freq=='default':
                plt.xlabel('Simulation Year')
                plt.title(f'{var}\n5-day Output')
            else:
                plt.xlabel(freq)
                plt.title(f'{var}\n{freq} ({resampleType})')
        if plotAnnual:
            fig[f'{var}_ann']=plt.figure(nF+1); nF+=1
            ax[f'{var}_ann']=plt.gca()
            fig[f'{var}_ann'].set_figwidth(1.25*fig[f'{var}_ann'].get_figwidth())
            plt.title(f'{var} ({resampleType})\nMean Annual Cycle')
            plt.ylabel(var)
            plt.xlabel('Month')
            plt.xlim([0.75,12.25])
            ax[f'{var}_ann'].set_xticks(range(1,13))
            ax[f'{var}_ann'].set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'])

        # loop through each runid and load variable if available
        for iR,rR in enumerate(runids):
            rfile=os.path.join(paths[iR],f'{rR}_{var}_timeseries.nc')
            if os.path.isfile(rfile):
                with xr.open_dataset(rfile,decode_times='False') as ds:
                    for iF,freq in enumerate(freqs2plot):
                        # loop through each plotting frequency and add to plot
                        # because values are not necessarily ordered in time, drop duplicates
                        if freq=='default':
                            ax[f'{var}_{freq}'].plot(year0[iR]+ds.drop_duplicates(dim='time',keep='last').sortby('time')['year'],
                                                     ds.drop_duplicates(dim='time',keep='last').sortby('time')[var],color=f'C{iR}',linewidth=1,label=rR)
                        else:
                            # need to resample to get desired mean value
                            if freq.lower() in ['jan','feb','mar','apr','may','jun','jul','aug','sep','oct','nov','dec']:
                                if resampleType=='min':
                                    rds=ds.drop_duplicates(dim='time',keep='last').sortby('time').resample(time='1ME',skipna=True).min()
                                elif resampleType=='max':
                                    rds=ds.drop_duplicates(dim='time',keep='last').sortby('time').resample(time='1ME',skipna=True).max()
                                else:
                                    rds=ds.drop_duplicates(dim='time',keep='last').sortby('time').resample(time='1ME',skipna=True).mean()
                                # indexes of specific month
                                iM=np.where(np.array(['jan','feb','mar','apr','may','jun','jul','aug','sep','oct','nov','dec'])==freq.lower())[0][0]
                                mnth_idxs=rds.groupby('time.month').groups[iM]
                                # Extract the desired month by selecting the relevant indices
                                rds=rds.isel(time=mnth_idxs)
                            else:
                                if resampleType=='min':
                                    rds=ds.drop_duplicates(dim='time',keep='last').sortby('time').resample(time=freq,skipna=True).min()
                                elif resampleType=='max':
                                    rds=ds.drop_duplicates(dim='time',keep='last').sortby('time').resample(time=freq,skipna=True).max()
                                else:
                                    rds=ds.drop_duplicates(dim='time',keep='last').sortby('time').resample(time=freq,skipna=True).mean()
                            ax[f'{var}_{freq}'].plot(year0[iR] + rds['year'].values,rds[var],color=f'C{iR}',linewidth=1,label=rR)
                            # ax[f'{var}_{freq}'].set_xlim([dt.datetime(2001,1,1),dt.datetime(2017+10,12,31)])
                            rds.close()

                    if plotAnnual:
                        annualCycle(ds,var,f'C{iR}',rR,ax[f'{var}_ann'])
        
        # add legends and finalize figures
        for iA in list(fig.keys()):
            ax[iA].legend(loc='center left', bbox_to_anchor=(1, 0.5))
            ax[iA].grid(visible=True)
            fig[iA].tight_layout()